# 3. Identity Governance

Identity governance answers the question: **"Do the right people have the right access to the right resources at the right time?"**

This is about ongoing management — not just granting access, but reviewing it, limiting it over time, and detecting anomalies.

## The four pillars of Entra ID Governance

| Capability | What it does | When to use it |
|------------|-------------|----------------|
| **Entitlement Management** | Bundles of resources ("access packages") that users can request | Onboarding, project teams, partner access |
| **Access Reviews** | Periodic certification that access is still needed | Quarterly reviews, compliance audits |
| **Privileged Identity Management (PIM)** | Just-in-time elevation for admin roles | Anyone with admin access |
| **Identity Protection** | Risk-based detection of compromised identities | Everyone (detects leaked credentials, impossible travel, etc.) |

---
## 1. Privileged Identity Management (PIM)

**Problem**: people get admin roles and keep them forever. A compromised admin account = total breach.

**PIM solution**: admin roles are **eligible**, not **active**. When someone needs admin access, they **activate** the role for a limited time (e.g., 4 hours), optionally requiring approval and MFA.

### How PIM works

```
Alice (eligible for Global Admin)
  │
  ├── Normal day: no admin permissions (Reader only)
  │
  └── Needs to change a policy:
        1. Opens PIM portal
        2. Requests "Global Admin" activation
        3. Provides justification: "Need to update CA policy for APAC"
        4. Completes MFA challenge
        5. Manager approves (if approval required)
        6. Role active for 4 hours
        7. Automatically deactivates
```

### Key PIM concepts for the exam

| Concept | Meaning |
|---------|----------|
| **Eligible assignment** | User *can* activate the role but doesn't have it by default |
| **Active assignment** | User has the role right now (permanent or time-limited) |
| **Activation** | The act of temporarily enabling an eligible role |
| **Justification** | Required text explaining *why* the role is needed |
| **Approval workflow** | Another person must approve the activation |
| **Maximum activation duration** | How long the role stays active (e.g., 8 hours max) |

**Exam tip**: PIM requires **Entra ID Premium P2** (or Entra ID Governance license).

In [1]:
import json
from datetime import datetime, timedelta

# Simulate PIM role assignments
PIM_ASSIGNMENTS = {
    'alice': {'role': 'Global Administrator', 'type': 'eligible', 'max_duration_hours': 4},
    'bob':   {'role': 'User Administrator',   'type': 'eligible', 'max_duration_hours': 8},
    'carol': {'role': 'Security Reader',      'type': 'active',   'permanent': True},
}

def activate_role(user: str, justification: str, mfa_done: bool, approved: bool) -> dict:
    assignment = PIM_ASSIGNMENTS.get(user)
    if not assignment:
        return {'result': '❌ No PIM assignment found'}
    if assignment['type'] == 'active':
        return {'result': f'ℹ️ {user} already has active "{assignment["role"]}" role'}
    
    checks = []
    checks.append(('Justification provided', bool(justification)))
    checks.append(('MFA completed', mfa_done))
    checks.append(('Approval granted', approved))
    
    all_passed = all(c[1] for c in checks)
    now = datetime.now()
    return {
        'user': user,
        'role': assignment['role'],
        'checks': [{'check': c[0], 'result': '✅' if c[1] else '❌'} for c in checks],
        'result': f'✅ ACTIVATED until {(now + timedelta(hours=assignment["max_duration_hours"])).strftime("%H:%M")}' if all_passed else '❌ DENIED',
        'justification': justification,
    }

print('=== Scenario 1: Proper activation ===')
print(json.dumps(activate_role('alice', 'Need to update CA policy for APAC rollout', True, True), indent=2))

print('\n=== Scenario 2: No MFA ===')
print(json.dumps(activate_role('alice', 'Need to update CA policy', False, True), indent=2))

print('\n=== Scenario 3: No justification ===')
print(json.dumps(activate_role('bob', '', True, True), indent=2))

print('\n=== Scenario 4: Already active ===')
print(json.dumps(activate_role('carol', 'n/a', True, True), indent=2))

=== Scenario 1: Proper activation ===
{
  "user": "alice",
  "role": "Global Administrator",
  "checks": [
    {
      "check": "Justification provided",
      "result": "\u2705"
    },
    {
      "check": "MFA completed",
      "result": "\u2705"
    },
    {
      "check": "Approval granted",
      "result": "\u2705"
    }
  ],
  "result": "\u2705 ACTIVATED until 05:32",
  "justification": "Need to update CA policy for APAC rollout"
}

=== Scenario 2: No MFA ===
{
  "user": "alice",
  "role": "Global Administrator",
  "checks": [
    {
      "check": "Justification provided",
      "result": "\u2705"
    },
    {
      "check": "MFA completed",
      "result": "\u274c"
    },
    {
      "check": "Approval granted",
      "result": "\u2705"
    }
  ],
  "result": "\u274c DENIED",
  "justification": "Need to update CA policy"
}

=== Scenario 3: No justification ===
{
  "user": "bob",
  "role": "User Administrator",
  "checks": [
    {
      "check": "Justification provided",
      "r

### Standing admin vs just-in-time admin

Below we simulate a breach of Alice's account under two setups: (1) she's a permanent Global Administrator (the old way), and (2) she's only *eligible* via PIM (the best-practice way).

In [2]:
def attacker_uses_stolen_credentials(user_state):
    if user_state['active_admin']:
        return '💥 Full tenant compromise — attacker has Global Admin right now'
    return '🛡️  Attacker has a user session but NO admin rights until PIM activation (blocked by MFA + approval)'

print('❌ BAD: Alice is a standing Global Admin')
print('  →', attacker_uses_stolen_credentials({'active_admin': True}))

print('\n✅ GOOD: Alice is *eligible* via PIM, not active by default')
print('  →', attacker_uses_stolen_credentials({'active_admin': False}))

print('\nPIM reduces the blast radius: standing admins ≈ 0, so a stolen password is not enough.')

❌ BAD: Alice is a standing Global Admin
  → 💥 Full tenant compromise — attacker has Global Admin right now

✅ GOOD: Alice is *eligible* via PIM, not active by default
  → 🛡️  Attacker has a user session but NO admin rights until PIM activation (blocked by MFA + approval)

PIM reduces the blast radius: standing admins ≈ 0, so a stolen password is not enough.


---
## 2. Access Reviews

**Problem**: users accumulate permissions over time ("permission creep"). Someone who changed teams 2 years ago still has access to the old team's SharePoint.

**Access reviews** are periodic checks where managers or resource owners confirm that each person's access is still needed.

| Setting | Options |
|---------|----------|
| **Reviewers** | Managers, group owners, self-review, specific users |
| **Frequency** | One-time, weekly, monthly, quarterly, annually |
| **Scope** | Group membership, app assignments, Entra/Azure roles |
| **If reviewer doesn't respond** | Remove access, approve, or no change |
| **Auto-apply results** | Automatically remove access for denied users |

### Exam tip

Access reviews can target: group members, app users, Entra role holders, and Azure role holders. They help with **compliance** (SOX, GDPR) and **least privilege**.

In [3]:
# Simulate an access review
GROUP_MEMBERS = [
    {'user': 'alice@contoso.com', 'added': '2024-01-15', 'last_used': '2026-04-10', 'department': 'Engineering'},
    {'user': 'bob@contoso.com',   'added': '2023-06-01', 'last_used': '2024-08-22', 'department': 'Marketing'},
    {'user': 'carol@contoso.com', 'added': '2025-11-20', 'last_used': '2026-04-15', 'department': 'Engineering'},
    {'user': 'dave@contoso.com',  'added': '2023-01-01', 'last_used': '2023-03-15', 'department': 'Former employee'},
]

print('=== Access Review: "Engineering Repo Access" group ===')
print(f'Reviewer: engineering-lead@contoso.com')
print(f'Auto-remove if no response in 14 days\n')

for member in GROUP_MEMBERS:
    last = datetime.strptime(member['last_used'], '%Y-%m-%d')
    days_since_use = (datetime.now() - last).days
    
    if member['department'] == 'Former employee':
        recommendation = '🔴 DENY — former employee, remove immediately'
    elif days_since_use > 365:
        recommendation = f'🟡 DENY — last used {days_since_use} days ago'
    elif member['department'] != 'Engineering':
        recommendation = f'🟡 REVIEW — department is {member["department"]}, not Engineering'
    else:
        recommendation = f'🟢 APPROVE — active user, correct department'
    
    print(f'{member["user"]}')
    print(f'  Added: {member["added"]}  Last used: {member["last_used"]}  Dept: {member["department"]}')
    print(f'  Recommendation: {recommendation}\n')

=== Access Review: "Engineering Repo Access" group ===
Reviewer: engineering-lead@contoso.com
Auto-remove if no response in 14 days

alice@contoso.com
  Added: 2024-01-15  Last used: 2026-04-10  Dept: Engineering
  Recommendation: 🟢 APPROVE — active user, correct department

bob@contoso.com
  Added: 2023-06-01  Last used: 2024-08-22  Dept: Marketing
  Recommendation: 🟡 DENY — last used 607 days ago

carol@contoso.com
  Added: 2025-11-20  Last used: 2026-04-15  Dept: Engineering
  Recommendation: 🟢 APPROVE — active user, correct department

dave@contoso.com
  Added: 2023-01-01  Last used: 2023-03-15  Dept: Former employee
  Recommendation: 🔴 DENY — former employee, remove immediately



---
## 3. Microsoft Entra ID Protection

ID Protection uses **machine learning** to detect risky sign-ins and risky users:

### Sign-in risk (per-session)

| Detection | What it means |
|-----------|---------------|
| Anonymous IP | Sign-in from Tor or VPN |
| Atypical travel | Impossible to travel between locations in time |
| Malware-linked IP | Sign-in from known malicious IP |
| Unfamiliar sign-in properties | Location, device, or browser never seen before |
| Password spray | Many accounts tried with common passwords |

### User risk (aggregated)

| Detection | What it means |
|-----------|---------------|
| Leaked credentials | User's password found in a dark-web dump |
| Threat intelligence | Microsoft detects the account is compromised |

### What you can do with risk signals

- Feed them into **Conditional Access** policies (e.g., "if sign-in risk is high, block")
- Require MFA or password change for risky users
- Investigate in the Identity Protection dashboard

**Exam tip**: ID Protection requires **Entra ID Premium P2**.

In [4]:
# Simulate ID Protection risk detection
SIGN_INS = [
    {'user': 'alice', 'ip': '203.0.113.1',   'location': 'Seattle', 'time': '09:00', 'device': 'laptop-alice',  'detections': []},
    {'user': 'alice', 'ip': '198.51.100.99', 'location': 'Moscow',  'time': '09:30', 'device': 'unknown-device', 'detections': ['atypical_travel', 'unfamiliar_device']},
    {'user': 'bob',   'ip': '10.0.0.50',     'location': 'Office',  'time': '08:45', 'device': 'laptop-bob',    'detections': []},
    {'user': 'carol', 'ip': '192.0.2.1',     'location': 'Tor exit','time': '02:00', 'device': 'unknown',       'detections': ['anonymous_ip', 'password_spray']},
]

RISK_SCORES = {
    'atypical_travel': 'high',
    'unfamiliar_device': 'medium',
    'anonymous_ip': 'high',
    'password_spray': 'high',
    'leaked_credentials': 'high',
}

print('=== Identity Protection: Sign-in Risk Analysis ===\n')
for si in SIGN_INS:
    if not si['detections']:
        risk = 'none'
        action = '✅ Allow'
    else:
        risk_levels = [RISK_SCORES.get(d, 'low') for d in si['detections']]
        risk = 'high' if 'high' in risk_levels else 'medium'
        action = '🚫 Block' if risk == 'high' else '🔐 Require MFA'
    
    print(f'User: {si["user"]}  Location: {si["location"]}  Device: {si["device"]}')
    print(f'  Detections: {si["detections"] or "none"}')
    print(f'  Risk level: {risk}')
    print(f'  Action: {action}\n')

=== Identity Protection: Sign-in Risk Analysis ===

User: alice  Location: Seattle  Device: laptop-alice
  Detections: none
  Risk level: none
  Action: ✅ Allow

User: alice  Location: Moscow  Device: unknown-device
  Detections: ['atypical_travel', 'unfamiliar_device']
  Risk level: high
  Action: 🚫 Block

User: bob  Location: Office  Device: laptop-bob
  Detections: none
  Risk level: none
  Action: ✅ Allow

User: carol  Location: Tor exit  Device: unknown
  Detections: ['anonymous_ip', 'password_spray']
  Risk level: high
  Action: 🚫 Block



---
## 4. Entitlement Management (access packages)

**Problem**: onboarding a new hire or a partner usually means a ticket storm — "add Bob to the Sales SharePoint, the CRM app, the Teams channel, the shared mailbox…".

**Entitlement Management** bundles those resources into an **access package** that users can *request* from a catalog. A policy controls who can request, who approves, and for how long.

### Access package anatomy

| Component | Example |
|-----------|---------|
| **Catalog** | "Sales team resources" |
| **Resources** | Sales SharePoint site, Salesforce app, Sales Teams channel |
| **Policy** | Who can request (internal users / specific partner tenants), who approves, how long access lasts, recurring review |

This is also how you grant **B2B guest** access cleanly: a partner requests the package, an internal owner approves, the guest account is auto-provisioned, and access auto-expires.

**Exam tip**: Entitlement Management is part of **Entra ID Governance** (requires Premium P2 or Entra ID Governance license).

In [5]:
# Mini simulation of an access package request
ACCESS_PACKAGES = {
    'sales-onboarding': {
        'resources': ['SharePoint: Sales site','App: Salesforce','Group: Sales-All'],
        'approver': 'sales-manager@contoso.com',
        'duration_days': 180,
        'requires_review': True,
    }
}

def request_access_package(user, package, justification, approved):
    pkg = ACCESS_PACKAGES[package]
    if not justification:
        return '❌ request rejected — justification required'
    if not approved:
        return f'⏳ pending approval by {pkg["approver"]}'
    return {
        'user': user,
        'package': package,
        'granted_resources': pkg['resources'],
        'expires_in_days': pkg['duration_days'],
        'review_required': pkg['requires_review'],
        'status': '✅ access granted (auto-expires, recurring review scheduled)',
    }

print('New hire requesting access:')
print(json.dumps(request_access_package('newhire@contoso.com','sales-onboarding','Joining the Sales team Monday', True), indent=2, default=str))

print('\nSame request with no justification:')
print(request_access_package('newhire@contoso.com','sales-onboarding','', True))

New hire requesting access:
{
  "user": "newhire@contoso.com",
  "package": "sales-onboarding",
  "granted_resources": [
    "SharePoint: Sales site",
    "App: Salesforce",
    "Group: Sales-All"
  ],
  "expires_in_days": 180,
  "review_required": true,
  "status": "\u2705 access granted (auto-expires, recurring review scheduled)"
}

Same request with no justification:
❌ request rejected — justification required


---
## Summary — Identity Governance

| Capability | Purpose | License |
|------------|---------|----------|
| **PIM** | Just-in-time admin access | Premium P2 |
| **Access Reviews** | Periodic verification of access | Premium P2 |
| **Entitlement Management** | Self-service access packages | Premium P2 |
| **ID Protection** | ML-based risk detection | Premium P2 |

### Exam cheat sheet

- PIM = **time-limited** admin roles, with justification + approval.
- Access reviews = **periodic** check that access is still needed.
- ID Protection = **automatic** detection of risky sign-ins and users.
- All three require **Premium P2** license.

**Next lab**: [03 — Azure Security Solutions](../../03-azure-security-solutions/)